# Part 2: The Stability Experiment (Temperature Stress Test)
- **The Concept**: Crisis systems require determinism.
- **Task**:
    - Load data/scenarios.txt (contains complex, ambiguous scenarios).
    - Experiment: For each scenario:
    - Run the CoT prompt 3 times with temperature=1.0 (Chaos Mode).
    - Run the CoT prompt 1 time with temperature=0.0 (Safe Mode).
  
- **Deliverable**: 
Print outputs and write a comment explaining specifically how the Chaos output drifted or hallucinated compared to the Safe output.

In [2]:
# Import libraries
import sys
sys.path.append('..')

from utils.token_utils import pick_encoding, count_text_tokens
from utils.logging_utils import log_llm_call
from utils.prompts import render
from utils.llm_client import LLMClient
from utils.router import pick_model
import tiktoken
import pandas as pd
from IPython.display import Markdown, display

In [26]:
# pick reasoning model
reasoning_model = pick_model('groq', 'reason')
client_reasoning = LLMClient('groq', reasoning_model)

In [11]:
with open("../data/Scenarios.txt", "r") as f:
    scenarios_text = f.read()
scenario_list = scenarios_text.strip().split("\n\n")

print("Scenarios loaded:")
print(scenario_list)
print(f"Total scenarios: {len(scenario_list)}")

Scenarios loaded:
['SCENARIO A: THE KANDY LANDSLIDE\nLocation: Hanthana Tea Factory\nDetails: "We are trapped. The access road is blocked. My uncle is stuck in the line rooms below, climbing a tree as water rises (Immediate Life Threat). However, inside the factory, we have a diabetic patient who has collapsed and needs insulin (Immediate Health Threat). We also have 40 people hungry."', 'SCENARIO B: THE GAMPAHA HOSPITAL\nLocation: Gampaha District General Hospital\nDetails: "Power has failed in the ICU. We have 3 patients on ventilators with battery backup for 2 hours (Critical). Meanwhile, flood water is entering the ground floor ward where 50 elderly patients are bedridden (High Risk). We have one generator truck arriving in 30 mins, but it can only power one section."']
Total scenarios: 2


In [35]:
results = []

for scenario in scenario_list:
    display(Markdown(f"### Scenario: <br>{scenario}..."))

    structured_response = {
        "scenario": scenario,
        "chaos_mode_results": [],
        "safe_mode_result": None
    }

    instructions='''
    Given the above scenario, analyse step-by-step:
    1. What is the most likely district in the message? If no specific location can be identified, respond with "None".
    2. What is the risk level?
    3. What is the immediate action required?
    4. Rank Each situation in the details section from highest to lowest priority. If no specific action is required, respond with "None".
    '''

    complete_prompt = f"{scenario}\n\n{instructions}"

    prompt_text, spec = render(
        'cot_reasoning.v1',
        role="Crisis message analyst",
        problem=complete_prompt
    )

    messages = [{ "role": "user", "content": prompt_text }]

    response = client_reasoning.chat(messages, temperature=0.0, max_tokens=spec.max_tokens)
    display(Markdown("### Safe Mode Result"))
    display(Markdown(response["text"]))
    log_llm_call('groq', reasoning_model, 'reason', response['latency_ms'], response['usage'])

    structured_response["safe_mode_result"] = response
    

    display(Markdown("### Chaos Mode Results"))

    for i in range(3):
        response_chaos = client_reasoning.chat(messages, temperature=1.0, max_tokens=spec.max_tokens)
        display(Markdown(f"#### Chaos Mode Run {i+1}"))
        display(Markdown(response_chaos["text"]))
        log_llm_call('groq', reasoning_model, 'reason', response_chaos['latency_ms'], response_chaos['usage'])
        structured_response["chaos_mode_results"].append(response_chaos)

    results.append(structured_response)

### Scenario: <br>SCENARIO A: THE KANDY LANDSLIDE
Location: Hanthana Tea Factory
Details: "We are trapped. The access road is blocked. My uncle is stuck in the line rooms below, climbing a tree as water rises (Immediate Life Threat). However, inside the factory, we have a diabetic patient who has collapsed and needs insulin (Immediate Health Threat). We also have 40 people hungry."...

### Safe Mode Result

**Reasoning Steps**

1. Identify the geographic reference: “Hanthana Tea Factory” is situated in the Kandy district of Sri Lanka.  
2. Evaluate the overall situation: multiple simultaneous threats (life‑threat, health‑threat, and basic‑needs crisis). This qualifies as a **High** risk level.  
3. Determine what must be done first: rescue the trapped uncle and deliver insulin to the collapsed diabetic—both are immediate life‑threat/health‑threat actions.  
4. Rank each sub‑scenario by urgency: 
   - Trapped uncle (water rising, climbing tree)  
   - Collapsed diabetic patient (needs insulin)  
   - Hunger of 40 people (longer‑term need but not immediately life‑threat).  

**Answer**

1. **District:** Kandy  
2. **Risk Level:** High  
3. **Immediate Action Required:** Initiate rescue of the trapped uncle and provide urgent medical care (insulin) for the collapsed diabetic patient.  
4. **Priority Ranking:**  
   1. Uncle stuck in the line rooms, water rising.  
   2. Diabetic patient collapsed needing insulin.  
   3. 40 people hungry.

### Chaos Mode Results

#### Chaos Mode Run 1

...

**Reasoning Steps**  
1. Identify the district from the header – “Gampaha District General Hospital”.  
2. Determine overall risk: ICU ventilator failure is *critical* (life‑support) while flood water is high risk to many elderly; overall situation is *Critical* because patient survival depends on immediate power restoration.  
3. Immediate action: Secure power for the ICU first (generator allocation and battery use); start flood containment and consider patient relocation from the affected ward.  
4. Rank situations by urgency:  
   - ICU power failure with ventilator‑battery backup (2 h).  
   - Flood water entering ground‑floor ward with 50 elderly.  
   - Arrival of a single generator truck in 30 min (action to be taken).

---

**Answer:**  
1. District: **Gampaha**  
2. Risk Level: **Critical**  
3. Immediate Action Required: **Deploy the generator to the ICU as soon as it arrives, use the battery backup until then, and initiate flood containment measures (e.g., sandbags, evacuation of vulnerable patients if feasible).**  
4. Prioritization: **ICU power failure > Flood water in ward > Generator arrival**

#### Chaos Mode Run 2

**Reasoning Outline**

1. Identify district from hospital name → Gampaha District.  
2. Combine risk descriptors: ICU “Critical” (2‑hour battery, ventilated patients) and ward “High Risk” (50 bedridden). The overall situation is dominated by a *critical* life‑support failure.  
3. Immediate need is to keep the ICU powered; after that, address the flooding of the ground‑floor ward.  
4. Rank the separate sub‑situations from most urgent to least urgent based on patient safety and resource constraints.

---

**Answer**

1. **District**: Gampaha  
2. **Risk level**: Critical  
3. **Immediate action required**: Allocate the generator to the ICU immediately to keep the ventilated patients alive; concurrently deploy a rapid‑response team to remove or protect the 50 elderly patients from the flood‑affected ground floor.  
4. **Priority ranking**:
   1. ICU power failure with ventilated patients (Critical).  
   2. Flood water entering ground floor ward with 50 bedridden patients (High Risk).  
   3. Arrival of the generator truck in 30 min (logistical support).

#### Chaos Mode Run 3

**Reasoning Steps**

1. Identify the location name mentioned – “Gampaha District General Hospital”.  
2. Determine overall risk: ICU ventilator failure is life‑threatening (critical), flood risk is high; the most severe condition dictates overall risk → **Critical**.  
3. Immediate action: Generator can power only one section; prioritize ICU to keep ventilators running.  
4. Rank situations: ICU first (critical), flood second (high).

---

**Answer**

1. **District:** Gampaha  
2. **Risk level:** Critical  
3. **Immediate action required:** Allocate the arriving generator to the ICU immediately to sustain ventilator power; concurrently arrange flood mitigation for the ward.  
4. **Priority ranking:**  
   1. Power failure in ICU with ventilated patients (Critical)  
   2. Flood water entering ground‑floor ward with bedridden elderly patients (High)

In [36]:
print("Experiment completed. Results:\n")
for res in results:
    print(f"Scenario: {res['scenario']}\n")
    print(f"Safe Mode Result: {res['safe_mode_result']}")
    print(f"Chaos Mode Results: {res['chaos_mode_results']}")
    print("-" * 50, "\n")

Experiment completed. Results:

Scenario: SCENARIO A: THE KANDY LANDSLIDE
Location: Hanthana Tea Factory
Details: "We are trapped. The access road is blocked. My uncle is stuck in the line rooms below, climbing a tree as water rises (Immediate Life Threat). However, inside the factory, we have a diabetic patient who has collapsed and needs insulin (Immediate Health Threat). We also have 40 people hungry."

Safe Mode Result: {'text': '**Reasoning Steps**\n\n1. Identify the geographic reference: “Hanthana Tea Factory” is situated in the Kandy district of Sri\u202fLanka.  \n2. Evaluate the overall situation: multiple simultaneous threats (life‑threat, health‑threat, and basic‑needs crisis). This qualifies as a **High** risk level.  \n3. Determine what must be done first: rescue the trapped uncle and deliver insulin to the collapsed diabetic—both are immediate life‑threat/health‑threat actions.  \n4. Rank each sub‑scenario by urgency: \n   - Trapped uncle (water rising, climbing tree)  \n 

# Discussion

When inspecting the outputs of each response, it can be identified that the core decision content is mostly stable but confidence and wording drifted in chaos mode.

All runs in both scenarios identified the district correctly and kept the same priorities such as life threatening/rescue as top. But in high-temp runs had different severity labels and explanations. 

Eg: Scenario A: severity output shifted from "High" to "Critical" risk while having the same prompt and no new evidence. 

Also, chaos responses expanded the priority ranking to add additional tasks such as unblocking the access road in scenario A. This was not strictly required by the prompt as well. 

The experiment suggests semantic drift is higher than factual hallucination in this scenarios. The model generally stayed grounded in the scenario facts, but varied in framing, strictness, and expressiveness when temperature increased.

# Conclusion
For the given crisis intelligence requirement, temperature=0.0 is the safer operating mode because it provides more deterministic, repeatable outputs and more stable risk labels.

temperature=1.0 may still produce plausible answers but it introduces variance in severity judgments and action phrasing, which is not ideal in critical production systems. Safe mode should be the default for production while chaos mode is better for exploratory analysis and system testing.